In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import duckdb
import pandas as pd
import seaborn as sns
import toml
from loguru import logger


In [ ]:
def stdout_filter(record):
    return record["level"].name == "INFO" and "ECHO" in record["message"]

def customise_logger(logger_filename="DuckDBManager.log", logger_size=10, filter_func=stdout_filter, level="INFO"):
    try:
        logger.remove(0) # Remove default output to sys.stderr
        logger.add(sys.stdout, filter=filter_func, level=level)     # Add sys.stdout logging of select messages
        logger.add(logger_filename, rotation=f"{logger_size} MB", level=level)  # logs will be written to file.log
        logger.info(f"ECHO: Logging to {logger_filename}: size = {logger_size} MB")
        logger.info("ECHO: Limited logging to stdout using `stdout_filter`")
    except Exception as e:
        logger.error(f"Error with customising logger: {e}")

In [ ]:
customise_logger()

In [ ]:
logger.info(f"ECHO - DuckDB version: {duckdb.__version__}")

In [ ]:
def check_local_duckdb_database():
    """
    Checks if a local DuckDB database exists. It returns the path to the database if it exists, and None otherwise.
    """
    DATA_DIR = Path("../data")
    EXPERIMENT_TYPE = "RPE"  # or CB
    assert DATA_DIR.exists()
    local_ddb = DATA_DIR / f"Experiments_{EXPERIMENT_TYPE}_2023_Q4.ddb"  # DuckDB database name

    logger.info(f"ECHO - Database file: {local_ddb.resolve()}")

    if local_ddb.exists():
        return local_ddb
    else:
        return None

In [ ]:
local_ddb = check_local_duckdb_database()

In [ ]:
con = duckdb.connect(str(local_ddb))

In [ ]:
con.sql("PRAGMA database_list;")

In [ ]:
df = con.sql("""
    SELECT e.ExperimentType, ev.EventElapsedTime
    FROM Events ev
    JOIN Trials t ON ev.TrialID = t.TrialID
    JOIN Experiments e ON t.ExperimentID = e.ExperimentID
             """).df()

In [ ]:
df["ElapsedTruncTime"] = df["EventElapsedTime"].apply(lambda t: min(t, 30))

In [ ]:
df["ScaledTime"] = df["EventElapsedTime"] * 0.01

In [ ]:
df

In [ ]:

def create_boxplot(df, category_col, value_col, category_order=None, title=None):
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots()
    sns.boxplot(x=category_col, y=value_col, data=df, order=category_order, showfliers=False, ax=ax)
    ax.set_title(title)
    plt.show()


In [ ]:
create_boxplot(df, "ExperimentType", "EventElapsedTime", title="1st plot")

In [ ]:
create_boxplot(df, "ExperimentType", "ElapsedTruncTime", title = "Trunc plot")

In [ ]:
category_order = ["RPE-A," "RPE-H", "RPE-E", "RPE-ER", "RPE-R"]

In [ ]:
create_boxplot(df, "ExperimentType", "ScaledTime", category_order=category_order, title="Scaled plot")